# Learn `src/ui/app.py` Step by Step

This notebook is a guided tour of the current Streamlit UI file.

Important: do not import `src/ui/app.py` directly in a notebook. Streamlit scripts execute top-to-bottom at import time. This notebook reads `app.py` as source text, shows each section, and tests the non-UI logic safely.

In [ ]:
from pathlib import Path
import ast
import inspect
import re
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src' / 'ui' / 'app.py').exists()), cwd)
src_path = project_root / 'src'
app_path = project_root / 'src' / 'ui' / 'app.py'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

source = app_path.read_text(encoding='utf-8', errors='replace')
lines = source.splitlines()
tree = ast.parse(source)

print('project_root:', project_root)
print('app_path:', app_path)
print('total_lines:', len(lines))

In [ ]:
def show_lines(start, end):
    end = min(end, len(lines))
    for n in range(start, end + 1):
        print(f'{n:4d}: {lines[n-1]}')

def top_level_defs():
    result = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            result.append({
                'name': node.name,
                'kind': type(node).__name__,
                'start': node.lineno,
                'end': getattr(node, 'end_lineno', node.lineno),
            })
    return result

def find_def(name):
    for item in top_level_defs():
        if item['name'] == name:
            return item
    raise KeyError(name)

def show_def(name, pad=2):
    item = find_def(name)
    print(f"{item['kind']} {name}: lines {item['start']}-{item['end']}")
    show_lines(max(1, item['start'] - pad), item['end'] + pad)

print('Top-level functions/classes:')
for item in top_level_defs():
    print(f"{item['start']:4d}-{item['end']:4d} {item['kind']:12s} {item['name']}")

## 1. Current Functionality Map

The current `app.py` includes these UI features:

- Streamlit page configuration
- persistent session state: `thread_id`, `chat_history`, `last_result`, `pending_hitl`
- `_config()` helper for LangGraph thread configuration
- `run_query(..., approved=False)` bridge to LangGraph
- `GraphResponse` display adapter
- `_render_response_meta()` shared metrics/expanders
- `_render_hitl_panel()` approval/reject flow for ticket creation
- sidebar settings, memory controls, example queries, and clear chat
- chat history rendering
- query submit flow and pending HITL detection

In [ ]:
feature_checks = {
    'pending_hitl session state': 'pending_hitl' in source,
    'approved argument': 'approved: bool = False' in source,
    'GraphResponse errors': 'self.errors' in source,
    'GraphResponse pending_action': 'self.pending_action' in source,
    'execution_ms metric': 'execution_ms' in source,
    'shared response meta renderer': 'def _render_response_meta' in source,
    'HITL panel renderer': 'def _render_hitl_panel' in source,
    'approve button': 'Approve' in source and 'approved   = True' in source,
    'reject button': 'Reject' in source,
}

for name, ok in feature_checks.items():
    print(f'{name:34s}', ok)
    assert ok

## 2. Imports and Page Config

The script adds `src/` to `sys.path`, imports the compiled graph, initial state builder, and global config. Then it sets Streamlit page metadata.

In [ ]:
show_lines(1, 27)

assert 'st.set_page_config' in source
assert 'from graph.graph import copilot_graph' in source
assert 'from graph.state import initial_state' in source

## 3. Session State

`st.session_state` is how Streamlit keeps state across script reruns. The new `pending_hitl` key stores an approval request when the graph asks for human confirmation before creating tickets.

In [ ]:
session_start = next(i for i, line in enumerate(lines, 1) if 'Session state' in line)
helpers_start = next(i for i, line in enumerate(lines, 1) if 'Helpers' in line)
show_lines(session_start, helpers_start - 1)

expected_keys = ['thread_id', 'chat_history', 'last_result', 'pending_hitl']
for key in expected_keys:
    assert key in source
    print('found session key:', key)

## 4. `_config()` and `run_query()`

`_config()` provides the LangGraph memory thread id. `run_query()` builds an `AgentState` and invokes the graph.

Current UI intent: `approved=True` is passed during HITL approval so the graph can create tickets on the second run.

In [ ]:
show_def('_config')
print('\n')
show_def('run_query')

In [ ]:
from graph.state import initial_state

sig = inspect.signature(initial_state)
print('initial_state signature:', sig)

sample_state = initial_state(
    query='Why did retention drop last month?',
    thread_id='notebook-ui-learning',
    user_id='streamlit-user',
    time_range='last_month',
)
sample_state['approved'] = True
sample_config = {'configurable': {'thread_id': sample_state['thread_id']}}

print('sample_state thread_id:', sample_state['thread_id'])
print('sample_state approved:', sample_state['approved'])
print('sample_config:', sample_config)

if 'approved' not in sig.parameters:
    print('WARNING: app.py passes approved=... into initial_state(), but initial_state does not currently accept that keyword.')
    print('Fix option: add approved: bool = False to initial_state(...) and set "approved": approved in the returned dict.')
else:
    print('PASS: initial_state accepts approved=... for HITL approval reruns.')

## 5. `GraphResponse`

`GraphResponse` converts the raw graph result dictionary into UI-friendly fields. The current version includes Day 15 fields:

- `errors`
- `pending_action`
- `execution_ms`

In [ ]:
show_def('GraphResponse')

In [ ]:
class GraphResponseForNotebook:
    def __init__(self, result: dict):
        self.summary = result.get('final_summary', '')
        self.intent = result.get('intent', 'unknown')
        self.confidence = result.get('confidence', 0.0)
        self.sources = result.get('sources', [])
        self.auto_tickets = result.get('auto_tickets', [])
        self.anomalies = result.get('anomalies', [])
        self.errors = result.get('errors', [])
        self.pending_action = result.get('pending_action')
        self.execution_ms = result.get('execution_ms', 0.0)
        self.success = bool(self.summary)
        self.agents_used = [r.get('agent', '') for r in result.get('agent_results', []) if r.get('agent')]
        self.data = next(
            (r.get('data', {}) for r in result.get('agent_results', []) if r.get('agent') == 'information_agent' and r.get('success')),
            {},
        )
        self.memory_turns = len(result.get('conversation_history', []))

mock_result = {
    'final_summary': 'Retention dropped because GRR fell below threshold.',
    'intent': 'metric_analysis',
    'confidence': 0.89,
    'sources': ['[MOCK] analytics.retention_metrics'],
    'auto_tickets': [],
    'anomalies': ['GRR below 85% threshold'],
    'errors': [{'node': 'metadata_node', 'error': 'example warning'}],
    'pending_action': {'message': 'Create Jira tickets?', 'anomalies': ['GRR below threshold'], 'products': ['retention'], 'count': 1},
    'execution_ms': 1234.5,
    'agent_results': [
        {'agent': 'information_agent', 'success': True, 'data': {'metrics': {'retention': {'gross_retention_rate': 81.65}}}},
        {'agent': 'knowledge_agent', 'success': True, 'data': {}},
    ],
    'conversation_history': [{'query': 'q', 'summary': 's'}],
}

response = GraphResponseForNotebook(mock_result)
print('success:', response.success)
print('agents_used:', response.agents_used)
print('pending_action:', response.pending_action)
print('execution_ms:', response.execution_ms)
print('errors:', response.errors)

assert response.success
assert response.pending_action['count'] == 1
assert response.execution_ms == 1234.5
assert response.errors[0]['node'] == 'metadata_node'

## 6. `_render_response_meta()`

This helper renders the shared response UI: confidence, agent count, intent, execution time, memory turns, anomalies, tickets, errors, raw metrics, sources, and agents used.

In [ ]:
show_def('_render_response_meta')

meta_expectations = ['Confidence', 'Agents', 'Intent', 'Exec ms', 'Memory', 'Raw metrics', 'Sources', 'Agents used']
for text in meta_expectations:
    print(text, text in source)
    assert text in source

## 7. `_render_hitl_panel()`

The HITL panel appears when `st.session_state.pending_hitl` is set. The approval path reruns the graph with `approved=True`; the reject path clears pending state.

In [ ]:
show_def('_render_hitl_panel')

hitl_expectations = [
    'st.session_state.pending_hitl',
    'Action requires your approval',
    'approved   = True',
    'Reject',
    'st.session_state.pending_hitl = None',
    'st.rerun()',
]
for text in hitl_expectations:
    print(text, text in source)
    assert text in source

In [ ]:
# Simulate the pending_hitl payload shape used by the UI.
pending_hitl = {
    'message': 'Create Jira tickets for detected anomalies?',
    'anomalies': ['GRR below 85% threshold', '36 accounts at risk'],
    'products': ['retention'],
    'count': 2,
    'query': 'Why did retention drop last month?',
    'time_range': 'last_month',
}

required_hitl_keys = ['message', 'anomalies', 'products', 'count', 'query', 'time_range']
for key in required_hitl_keys:
    print(key, pending_hitl[key])
    assert key in pending_hitl

## 8. Header and Sidebar

The sidebar controls mock mode, time range, memory/thread reset, example query selection, and chat clearing. The current reset paths also clear `pending_hitl`.

In [ ]:
header_start = next(i for i, line in enumerate(lines, 1) if line.startswith('st.title'))
chat_history_start = next(i for i, line in enumerate(lines, 1) if 'Chat history' in line)
show_lines(header_start, chat_history_start - 1)

sidebar_features = ['Mock Mode', 'Time Range', 'Memory', 'New conversation', 'Try these', 'Clear chat', 'pending_query']
for feature in sidebar_features:
    print(feature, feature in source)
    assert feature in source

## 9. Chat History Rendering

The script redraws saved chat messages first. The current history path renders compact inline metadata from `msg['meta']`.

In [ ]:
hitl_panel_call = next(i for i, line in enumerate(lines, 1) if line.strip() == '_render_hitl_panel()')
show_lines(chat_history_start, hitl_panel_call)

fake_history_meta = {
    'confidence': 0.89,
    'agents_used': ['information_agent', 'knowledge_agent'],
    'intent': 'metric_analysis',
    'memory_turns': 1,
    'anomalies': ['GRR below threshold'],
    'auto_tickets': [],
    'data': {},
    'sources': [],
    'errors': [],
    'execution_ms': 1234.5,
}
assert fake_history_meta['execution_ms'] > 0
assert len(fake_history_meta['agents_used']) == 2

## 10. Query Input Flow

The live query flow appends the user message, runs the graph, renders the response, stores pending HITL if needed, then saves assistant metadata.

In [ ]:
query_start = next(i for i, line in enumerate(lines, 1) if 'Query input' in line)
show_lines(query_start, len(lines))

query_flow_snippets = [
    'st.chat_input',
    'raw_result = run_query',
    'response   = GraphResponse',
    'response.pending_action',
    'pending_hitl',
    'execution_ms',
    'st.session_state.last_result',
]
for snippet in query_flow_snippets:
    print(f'{snippet:32s}', snippet in source)
    assert snippet in source

## 11. Optional Full Graph Run

This can call the configured LLM provider and may create Redis cache entries. Keep it disabled while you are only learning the UI file.

In [ ]:
RUN_ACTUAL_GRAPH_QUERY = False

if RUN_ACTUAL_GRAPH_QUERY:
    from graph.graph import copilot_graph
    result = copilot_graph.invoke(sample_state, config=sample_config)
    response = GraphResponseForNotebook(result)
    print('intent:', response.intent)
    print('agents:', response.agents_used)
    print('pending_action:', response.pending_action)
    print('execution_ms:', response.execution_ms)
    print('summary:', response.summary[:1000])
else:
    print('Skipped. Set RUN_ACTUAL_GRAPH_QUERY = True to run this cell.')

## 12. End-To-End Mental Model

```text
User enters query or clicks example
  -> Streamlit reruns app.py
  -> query is appended to chat_history
  -> run_query builds AgentState and invokes LangGraph
  -> GraphResponse extracts summary, agents, metrics, errors, pending_action
  -> _render_response_meta displays metrics, anomalies, tickets, errors, raw data
  -> if pending_action exists, pending_hitl is saved and the page reruns
  -> _render_hitl_panel shows Approve / Reject controls
  -> Approve reruns graph with approved=True
  -> assistant response metadata is saved in chat_history
```

Launch locally:

```powershell
uv run streamlit run src/ui/app.py
```

Launch with Docker:

```powershell
docker compose up -d --build app redis
docker compose logs -f app
```

## 13. Debugging Checklist

- Do `config.settings`, `graph.state`, and `graph.graph` import?
- Does `initial_state` accept every keyword that `app.py` passes?
- Does `GraphResponse` include `errors`, `pending_action`, and `execution_ms`?
- Does `_render_response_meta` receive a fully initialized `GraphResponse` object?
- Does `pending_hitl` clear when starting a new conversation or clearing chat?
- If a ticket approval appears, does approve rerun with `approved=True`?
- If Redis looks empty, did the query route to `information`, `knowledge`, or `metadata`?
- Use `docker compose logs -f app` to see cache hit/miss and graph errors.